# Hospital data analysis

**Original coursework: Abhinav Varma & Devika Rani Sanaboyina (DATA 602).** The original notebooks and PDFs are preserved at the repository root. See `docs/DATA_PROVENANCE.md` for source versions and artifact checksums.

The analysis follows CRISP-DM to describe capacity and resource patterns and evaluate traditional machine learning models for reported expenses. The outcomes describe reported resources and expenses; they do not measure hospital efficiency, clinical quality, or future demand.

The source is California HCAI/OSHPD hospital financial reporting data for 2012–2017. Each long-format row represents one measure (`Variable`) for a facility reporting period. Fetch the inputs with `python scripts/fetch_data.py --historical-cleaned` before execution. Each notebook runs independently after the inputs are available.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "hospital_ml").is_dir())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Image
from hospital_ml.data import read_raw, prepare_data, write_json
SEED = 42


In [2]:
from hospital_ml.data import sha256
raw = read_raw()
frame, audit = prepare_data(raw)
manifest = json.loads((ROOT / "data/source_manifest.json").read_text())
assert sha256(ROOT / manifest["raw"]["path"]) == manifest["raw"]["sha256"]
write_json(ROOT / "reports/data/audit.json", audit)
display(pd.Series({k: v for k,v in audit.items() if not isinstance(v,dict)}, name="Observed"))
display(pd.DataFrame({"missing_values": audit["missing_values"]}))

raw_rows                                578350
raw_columns                                 35
exact_full_schema_duplicates_removed         0
missing_identity_rows_excluded               0
prepared_reporting_rows                 578350
facilities                                 472
facility_reporting_periods                2690
variables                                  228
Name: Observed, dtype: int64

,missing_values
FAC_NO,0
FAC_NAME,0
BEG_DATE,0
END_DATE,0
COUNTY,430
DAY_PER,0
BED_LIC,10320
BED_AVL,6235
BED_STF,4945
Variable,0


## Data preparation

Retain facility IDs and report dates. Remove exact duplicate rows before selecting columns, so equal values from different reporting periods remain distinct. Convert suppressed or invalid numeric capacity values to missing and retain missing COUNTY values for imputation within the model pipeline. Handle missing targets within the supervised task.

Learn imputation values, clipping bounds, scaling parameters, categorical vocabularies, and classification cutoffs from training facilities only. Retain extreme target values and compare development results with and without predictor clipping.

In [3]:
from hospital_ml.clustering import build_capacity_table
capacity = build_capacity_table(frame)
capacity_columns = ["BED_LIC", "BED_AVL", "BED_STF", "DAY_PER"]
display(capacity[capacity_columns].describe().T)
display(capacity[capacity_columns[:3]].corr())
capacity[capacity_columns].describe().T.to_csv(ROOT / "reports/data/capacity_description.csv")
capacity[capacity_columns[:3]].corr().to_csv(ROOT / "reports/data/capacity_correlations.csv")
coverage = frame.groupby("Variable").agg(reporting_rows=("Amount", "size"), nonmissing_amount=("Amount", "count"))
coverage.to_csv(ROOT / "reports/data/variable_coverage.csv")
print("Reporting periods:", len(capacity), "Distinct facilities:", capacity.FAC_NO.nunique())

,count,mean,std,min,25%,50%,75%,max
BED_LIC,2642.0,204.697199,175.924887,0.0,68.5,156.0,298.0,991.0
BED_AVL,2661.0,197.518602,169.170909,0.0,65.0,153.0,287.0,991.0
BED_STF,2667.0,133.997375,131.253683,0.0,41.0,95.0,187.0,989.0
DAY_PER,2690.0,360.442379,31.999986,22.0,365.0,365.0,366.0,399.0


,BED_LIC,BED_AVL,BED_STF
BED_LIC,1.000000,0.982091,0.917807
BED_AVL,0.982091,1.000000,0.927375
BED_STF,0.917807,0.927375,1.000000


Reporting periods: 2690 Distinct facilities: 472


## Measure definitions

The official dictionary defines licensed beds as **end-of-period** counts and available/staffed beds as **period averages** (dictionary printed page 3). `BED_STF` counts staffed beds, not people. Capacity ratios summarize different reporting conventions and can exceed one; they are not validated utilization rates.

`Amount` includes dollars, counts, hours, and ratios depending on `Variable`. Capacity clustering excludes `Amount`, and supervised modeling uses only the prespecified `TOT_OP_EXP` measure (total operating expenses; dictionary printed page 7). Strong bed correlations motivate licensed capacity plus two descriptive availability shares instead of weighting three correlated bed counts independently.

In [4]:
from hospital_ml.data import historical_cleaning, LEGACY_COLUMNS, sha256
cleaned, steps = historical_cleaning(raw)
original_path = ROOT / "data/historical/original_cleaned.csv"
match = cleaned.equals(pd.read_csv(original_path)[LEGACY_COLUMNS]) if original_path.exists() else None
result = {"steps": steps, "observed_cleaned_rows": len(cleaned), "values_match_downloaded_cleaned": match,
          "raw_sha256": sha256(ROOT / "data/raw/hospital_2012_2017.csv"),
          "cleaned_sha256": sha256(original_path) if original_path.exists() else None}
write_json(ROOT / "reports/historical/cleaning_replay.json", result)
display(pd.DataFrame(steps))
print("Exact eight-column match to original shared cleaned CSV:", match)

,step,rows,removed
0,raw,578350,NaN
1,drop_missing_selected_columns,577920,NaN
2,historical_projected_deduplication,553022,NaN
3,drop_non_numeric_values,542702,NaN
4,historical_sequential_iqr_BED_LIC,531435,11267.0
5,historical_sequential_iqr_BED_AVL,525845,5590.0
6,historical_sequential_iqr_BED_STF,513363,12482.0


Exact eight-column match to original shared cleaned CSV: True


The coursework cleaning procedure is retained as a separate data-lineage check. The updated models retain report dates and missing predictors, learn imputation from training data, and represent capacity at the facility-period level. The coursework subset is not used for the updated evaluation.

Source URLs, dictionary references, and checksums are recorded in `data/source_manifest.json`. Continue with `02_capacity_clustering.ipynb` and `03_supervised_evaluation.ipynb`.